
# Task 1 — Historical Average (HA) Baseline

This notebook implements the Historical Average baseline for the user's pre-windowed TraffiDent-style dataset.

## Dataset format

Each row has 390 columns:

- 384 input columns = 24 timesteps × 16 features
- 6 target columns = traffic flow at t+1 through t+6

The 16 features repeat in the same order for every timestep:

1. total_flow
2. precipitation
3. temperature_2m
4. wind_gusts_10m
5. relative_humidity
6. impact_sequence_hour
7. is_major_incident
8. is_holiday
9. hour_sin
10. hour_cos
11. is_weekend
12. feat_12
13. incident_type
14. lane_count
15. road_functional_hierarchy
16. distance_to_intersection

Forecast horizons evaluated:

- 1 hour
- 3 hours
- 6 hours

Three HA cases are included:

1. HA-General
2. HA-BinaryConditioned
3. HA-TypeConditioned

The notebook reports MAE, RMSE, and MAPE on:

- all validation/test samples
- incident-only validation/test samples

Note: the data are already Z-score normalized per station, so metrics are reported on the normalized scale unless inverse-scaling parameters are available.


In [14]:

# 1. Imports

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 2. Configuration

In [15]:

# Update these paths
TRAIN_PATH = "/Users/andishahifahmuthahharah/Downloads/traffic_data_24_6/train_24_6.csv"
VAL_PATH   = "/Users/andishahifahmuthahharah/Downloads/traffic_data_24_6/val_24_6.csv"
TEST_PATH  = "/Users/andishahifahmuthahharah/Downloads/traffic_data_24_6/test_24_6.csv"

N_TIMESTEPS = 24
N_FEATURES = 16
N_INPUT_COLUMNS = N_TIMESTEPS * N_FEATURES
N_TARGET_COLUMNS = 6
EXPECTED_TOTAL_COLUMNS = N_INPUT_COLUMNS + N_TARGET_COLUMNS

HORIZONS = [1, 3, 6]

TOTAL_FLOW_IDX = 0
IMPACT_SEQUENCE_HOUR_IDX = 5
INCIDENT_TYPE_IDX = 12

MIN_CLASS_COUNT = 5

OUTPUT_DIR = Path("ha_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Expected total columns:", EXPECTED_TOTAL_COLUMNS)
print("Output directory:", OUTPUT_DIR.resolve())


Expected total columns: 390
Output directory: /Users/andishahifahmuthahharah/Downloads/ha_results


## 3. Feature and incident metadata

In [16]:

FEATURE_NAMES = [
    "total_flow",
    "precipitation",
    "temperature_2m",
    "wind_gusts_10m",
    "relative_humidity",
    "impact_sequence_hour",
    "is_major_incident",
    "is_holiday",
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "feat_12",
    "incident_type",
    "lane_count",
    "road_functional_hierarchy",
    "distance_to_intersection",
]

INCIDENT_TYPE_MAP = {
    0: "NO_INCIDENT",
    1: "CRASH",
    2: "BREAKDOWN",
    3: "HAZARD",
    4: "ROADWORK",
    5: "TRAFFIC_CONTROL",
    6: "ADVERSE_WEATHER",
    7: "EVENT",
    8: "OTHERS",
}

for idx, name in enumerate(FEATURE_NAMES, start=1):
    print(f"{idx:2d}. {name}")


 1. total_flow
 2. precipitation
 3. temperature_2m
 4. wind_gusts_10m
 5. relative_humidity
 6. impact_sequence_hour
 7. is_major_incident
 8. is_holiday
 9. hour_sin
10. hour_cos
11. is_weekend
12. feat_12
13. incident_type
14. lane_count
15. road_functional_hierarchy
16. distance_to_intersection


## 4. Load the headerless files

In [17]:

def load_headerless_matrix(path, split_name):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"{split_name}: file not found: {path.resolve()}\n"
            "Update TRAIN_PATH, VAL_PATH, and TEST_PATH."
        )

    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path, header=None)
    elif suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(path, header=None)
    elif suffix in [".parquet", ".pq"]:
        df = pd.read_parquet(path)
    else:
        raise ValueError(
            f"{split_name}: unsupported extension {suffix}"
        )

    if df.shape[1] != EXPECTED_TOTAL_COLUMNS:
        raise ValueError(
            f"{split_name}: expected {EXPECTED_TOTAL_COLUMNS} columns, "
            f"but found {df.shape[1]}."
        )

    df = df.apply(pd.to_numeric, errors="coerce")

    missing = int(df.isna().sum().sum())
    if missing > 0:
        raise ValueError(
            f"{split_name}: found {missing} missing or non-numeric values."
        )

    return df


train_raw = load_headerless_matrix(TRAIN_PATH, "train")
val_raw   = load_headerless_matrix(VAL_PATH, "validation")
test_raw  = load_headerless_matrix(TEST_PATH, "test")

print("Train shape     :", train_raw.shape)
print("Validation shape:", val_raw.shape)
print("Test shape      :", test_raw.shape)

display(train_raw.head())


Train shape     : (603865, 390)
Validation shape: (206425, 390)
Test shape      : (187105, 390)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389
0,-0.751703,0.652621,-0.208682,-1.176887,0.583787,-1.0,0.0,0.0,0.000000,1.000000,0.0,1.0,0.0,2.0,5.0,0.0,-0.760985,0.652621,-0.208682,-1.176887,0.583787,-1.0,0.0,0.0,0.258819,0.965926,0.0,1.0,0.0,2.0,5.0,0.0,-0.758665,0.579785,-0.208682,-1.482412,0.744338,-1.0,0.0,0.0,0.500000,8.660254e-01,0.0,1.0,0.0,2.0,5.0,0.0,-0.767947,0.523134,...,-1.226953,-1.0,0.0,0.0,-0.707107,0.707107,0.0,1.0,0.0,2.0,5.0,0.0,-0.742421,1.235313,-0.208682,-0.321415,-0.782542,-1.0,0.0,0.0,-0.500000,0.866025,0.0,1.0,0.0,2.0,5.0,0.0,-0.758665,1.032989,-0.208682,0.320189,-0.224109,-1.0,0.0,0.0,-0.258819,0.965926,0.0,1.0,0.0,2.0,5.0,0.0,-0.742421,-0.756344,-0.747062,-0.740101,-0.705293,-0.626396
1,-0.760985,0.652621,-0.208682,-1.176887,0.583787,-1.0,0.0,0.0,0.258819,0.965926,0.0,1.0,0.0,2.0,5.0,0.0,-0.758665,0.579785,-0.208682,-1.482412,0.744338,-1.0,0.0,0.0,0.500000,0.866025,0.0,1.0,0.0,2.0,5.0,0.0,-0.767947,0.523134,-0.208682,-1.421307,0.859826,-1.0,0.0,0.0,0.707107,7.071068e-01,0.0,1.0,0.0,2.0,5.0,0.0,-0.765627,0.458391,...,-0.782542,-1.0,0.0,0.0,-0.500000,0.866025,0.0,1.0,0.0,2.0,5.0,0.0,-0.758665,1.032989,-0.208682,0.320189,-0.224109,-1.0,0.0,0.0,-0.258819,0.965926,0.0,1.0,0.0,2.0,5.0,0.0,-0.742421,0.822573,-0.208682,0.992345,0.386656,-1.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,2.0,5.0,0.0,-0.756344,-0.747062,-0.740101,-0.705293,-0.626396,-0.547499
2,-0.758665,0.579785,-0.208682,-1.482412,0.744338,-1.0,0.0,0.0,0.500000,0.866025,0.0,1.0,0.0,2.0,5.0,0.0,-0.767947,0.523134,-0.208682,-1.421307,0.859826,-1.0,0.0,0.0,0.707107,0.707107,0.0,1.0,0.0,2.0,5.0,0.0,-0.765627,0.458391,-0.208682,-1.360202,0.977939,-1.0,0.0,0.0,0.866025,5.000000e-01,0.0,1.0,0.0,2.0,5.0,0.0,-0.763306,0.434112,...,-0.224109,-1.0,0.0,0.0,-0.258819,0.965926,0.0,1.0,0.0,2.0,5.0,0.0,-0.742421,0.822573,-0.208682,0.992345,0.386656,-1.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,2.0,5.0,0.0,-0.756344,0.701179,-0.208682,1.206212,0.617979,-1.0,0.0,0.0,0.258819,0.965926,0.0,0.0,0.0,2.0,5.0,0.0,-0.747062,-0.740101,-0.705293,-0.626396,-0.547499,-0.542858
3,-0.767947,0.523134,-0.208682,-1.421307,0.859826,-1.0,0.0,0.0,0.707107,0.707107,0.0,1.0,0.0,2.0,5.0,0.0,-0.765627,0.458391,-0.208682,-1.360202,0.977939,-1.0,0.0,0.0,0.866025,0.500000,0.0,1.0,0.0,2.0,5.0,0.0,-0.763306,0.434112,-0.208682,-1.451860,1.047053,-1.0,0.0,0.0,0.965926,2.588190e-01,0.0,1.0,0.0,2.0,5.0,0.0,-0.749383,0.417926,...,0.386656,-1.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,2.0,5.0,0.0,-0.756344,0.701179,-0.208682,1.206212,0.617979,-1.0,0.0,0.0,0.258819,0.965926,0.0,0.0,0.0,2.0,5.0,0.0,-0.747062,0.620250,-0.208682,1.450633,0.712726,-1.0,0.0,0.0,0.500000,0.866025,0.0,0.0,0.0,2.0,5.0,0.0,-0.740101,-0.705293,-0.626396,-0.547499,-0.542858,-0.573025
4,-0.765627,0.458391,-0.208682,-1.360202,0.977939,-1.0,0.0,0.0,0.866025,0.500000,0.0,1.0,0.0,2.0,5.0,0.0,-0.763306,0.434112,-0.208682,-1.451860,1.047053,-1.0,0.0,0.0,0.965926,0.258819,0.0,1.0,0.0,2.0,5.0,0.0,-0.749383,0.417926,-0.208682,-1.696280,1.081868,-1.0,0.0,0.0,1.000000,6.123234e-17,0.0,1.0,0.0,2.0,5.0,0.0,-0.756344,0.571692,...,0.617979,-1.0,0.0,0.0,0.258819,0.965926,0.0,0.0,0.0,2.0,5.0,0.0,-0.747062,0.620250,-0.208682,1.450633,0.712726,-1.0,0.0,0.0,0.500000,0.866025,0.0,0.0,0.0,2.0,5.0,0.0,-0.740101,0.547413,-0.208682,1.756158,0.644414,-1.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,2.0,5.0,0.0,-0.705293,-0.626396,-0.547499,-0.542858,-0.573025,-0.533576


## 5. Reshape the 390-column matrix

In [18]:

def extract_arrays(df, split_name):
    values = df.to_numpy(dtype=np.float32)

    X = values[:, :N_INPUT_COLUMNS].reshape(
        -1, N_TIMESTEPS, N_FEATURES
    )

    y_all = values[:, N_INPUT_COLUMNS:]

    # t+1, t+3, t+6
    y = y_all[:, [0, 2, 5]]

    flow_history = X[:, :, TOTAL_FLOW_IDX]

    # Incident is active when impact_sequence_hour >= 0
    impact_at_t = X[:, -1, IMPACT_SEQUENCE_HOUR_IDX]
    incident_binary = (impact_at_t >= 0).astype(np.int64)

    # Incident category at the anchor timestep
    incident_type = np.rint(
        X[:, -1, INCIDENT_TYPE_IDX]
    ).astype(np.int64)

    # Non-incident samples must use category 0
    incident_type = np.where(
        incident_binary == 0,
        0,
        incident_type
    )

    valid_codes = np.array(list(INCIDENT_TYPE_MAP.keys()))
    invalid_mask = ~np.isin(incident_type, valid_codes)

    if invalid_mask.any():
        invalid_values = np.unique(incident_type[invalid_mask])
        raise ValueError(
            f"{split_name}: invalid incident type values: {invalid_values}"
        )

    return {
        "X": X,
        "y": y,
        "flow_history": flow_history,
        "incident_binary": incident_binary,
        "incident_type": incident_type,
    }


train = extract_arrays(train_raw, "train")
val   = extract_arrays(val_raw, "validation")
test  = extract_arrays(test_raw, "test")

print("X train:", train["X"].shape)
print("y train:", train["y"].shape)
print("Flow history:", train["flow_history"].shape)


X train: (603865, 24, 16)
y train: (603865, 3)
Flow history: (603865, 24)


## 6. Dataset audit

In [19]:

def audit_split(data, split_name):
    type_counts = pd.Series(
        data["incident_type"]
    ).value_counts().sort_index()

    row = {
        "split": split_name,
        "samples": len(data["y"]),
        "incident_rate": float(data["incident_binary"].mean()),
        "flow_mean": float(data["flow_history"].mean()),
        "flow_std": float(data["flow_history"].std()),
        "target_mean": float(data["y"].mean()),
        "target_std": float(data["y"].std()),
    }

    for code, label in INCIDENT_TYPE_MAP.items():
        row[f"n_{label}"] = int(type_counts.get(code, 0))

    return row


audit_df = pd.DataFrame([
    audit_split(train, "train"),
    audit_split(val, "validation"),
    audit_split(test, "test"),
])

display(audit_df)

print("Train incident distribution:")
display(
    pd.Series(train["incident_type"])
    .map(INCIDENT_TYPE_MAP)
    .value_counts()
    .rename_axis("incident_type")
    .reset_index(name="count")
)


,split,samples,incident_rate,flow_mean,flow_std,target_mean,target_std,n_NO_INCIDENT,n_CRASH,n_BREAKDOWN,n_HAZARD,n_ROADWORK,n_TRAFFIC_CONTROL,n_ADVERSE_WEATHER,n_EVENT,n_OTHERS
0,train,603865,0.002974,-0.000218,0.999669,0.001636,1.000486,602069,396,629,85,146,167,59,314,0
1,validation,206425,0.002684,0.221017,1.191085,0.222605,1.193259,205871,132,213,25,49,34,6,95,0
2,test,187105,0.003298,0.145483,1.167395,0.142942,1.166631,186488,134,235,34,27,75,14,94,4


Train incident distribution:


,incident_type,count
0,NO_INCIDENT,602069
1,BREAKDOWN,629
2,CRASH,396
3,EVENT,314
4,TRAFFIC_CONTROL,167
5,ROADWORK,146
6,HAZARD,85
7,ADVERSE_WEATHER,59



## 7. Historical Average model

For each sample:

\[
HA_i = \frac{1}{24}\sum_{k=1}^{24} flow_{i,k}
\]

Because the supplied matrix has no explicit station ID or raw timestamp, the appropriate HA is the average of the 24-hour historical flow window.

To distinguish the three requested cases:

- General uses a horizon-specific train residual correction.
- Binary uses a separate train residual correction for incident and non-incident samples.
- Type-conditioned uses a separate train residual correction for each incident category.

All corrections are fitted using train data only.


In [20]:

class HistoricalAverageModel:
    def __init__(self, mode="general", min_class_count=5):
        valid_modes = {"general", "binary", "type"}
        if mode not in valid_modes:
            raise ValueError(f"mode must be one of {valid_modes}")

        self.mode = mode
        self.min_class_count = min_class_count
        self.global_correction = None
        self.class_corrections = {}
        self.class_counts = {}

    @staticmethod
    def base_average(flow_history):
        return flow_history.mean(axis=1)

    def fit(self, data):
        base = self.base_average(data["flow_history"])
        residuals = data["y"] - base[:, None]

        self.global_correction = residuals.mean(axis=0)

        if self.mode == "general":
            return self

        labels = (
            data["incident_binary"]
            if self.mode == "binary"
            else data["incident_type"]
        )

        for label in np.unique(labels):
            mask = labels == label
            count = int(mask.sum())
            self.class_counts[int(label)] = count

            if count >= self.min_class_count:
                self.class_corrections[int(label)] = (
                    residuals[mask].mean(axis=0)
                )

        return self

    def predict(self, data):
        base = self.base_average(data["flow_history"])
        pred = base[:, None] + self.global_correction[None, :]

        if self.mode == "general":
            return pred

        labels = (
            data["incident_binary"]
            if self.mode == "binary"
            else data["incident_type"]
        )

        for label, correction in self.class_corrections.items():
            mask = labels == label
            pred[mask] = base[mask, None] + correction[None, :]

        return pred


## 8. Evaluation metrics

In [21]:

def safe_mape(y_true, y_pred, epsilon=1e-6):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.abs(y_true) > epsilon
    if mask.sum() == 0:
        return np.nan

    return float(
        np.mean(
            np.abs(
                (y_true[mask] - y_pred[mask]) /
                y_true[mask]
            )
        ) * 100
    )


def subset_data(data, mask):
    return {
        key: value[mask]
        for key, value in data.items()
    }


def evaluate(data, predictions, model_name, subset_name):
    rows = []

    for j, horizon in enumerate(HORIZONS):
        y_true = data["y"][:, j]
        y_pred = predictions[:, j]

        rows.append({
            "model": model_name,
            "subset": subset_name,
            "horizon_hour": horizon,
            "n_samples": len(y_true),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "MAPE_percent": safe_mape(y_true, y_pred),
        })

    return pd.DataFrame(rows)


## 9. Run the three HA experiments

In [22]:

EXPERIMENTS = {
    "HA-General": "general",
    "HA-BinaryConditioned": "binary",
    "HA-TypeConditioned": "type",
}

metrics_list = []
predictions_list = []
fitted_models = {}

for model_name, mode in EXPERIMENTS.items():
    print(f"Running {model_name}...")

    model = HistoricalAverageModel(
        mode=mode,
        min_class_count=MIN_CLASS_COUNT
    ).fit(train)

    fitted_models[model_name] = model

    # Validation
    val_pred = model.predict(val)
    metrics_list.append(
        evaluate(val, val_pred, model_name, "validation_all")
    )

    val_incident_mask = val["incident_binary"] == 1
    if val_incident_mask.any():
        metrics_list.append(
            evaluate(
                subset_data(val, val_incident_mask),
                val_pred[val_incident_mask],
                model_name,
                "validation_incident_only"
            )
        )

    # Test
    test_pred = model.predict(test)
    metrics_list.append(
        evaluate(test, test_pred, model_name, "test_all")
    )

    test_incident_mask = test["incident_binary"] == 1
    if test_incident_mask.any():
        metrics_list.append(
            evaluate(
                subset_data(test, test_incident_mask),
                test_pred[test_incident_mask],
                model_name,
                "test_incident_only"
            )
        )

    pred_df = pd.DataFrame({
        "sample_index": np.arange(len(test["y"])),
        "model": model_name,
        "incident_binary": test["incident_binary"],
        "incident_type_code": test["incident_type"],
        "incident_type": [
            INCIDENT_TYPE_MAP[int(x)]
            for x in test["incident_type"]
        ],
        "historical_average_24h": (
            test["flow_history"].mean(axis=1)
        ),
    })

    for j, horizon in enumerate(HORIZONS):
        pred_df[f"actual_t+{horizon}"] = test["y"][:, j]
        pred_df[f"pred_t+{horizon}"] = test_pred[:, j]

    predictions_list.append(pred_df)

metrics_df = pd.concat(metrics_list, ignore_index=True)
predictions_df = pd.concat(predictions_list, ignore_index=True)

display(
    metrics_df.sort_values(
        ["subset", "horizon_hour", "MAE"]
    ).reset_index(drop=True)
)


Running HA-General...
Running HA-BinaryConditioned...
Running HA-TypeConditioned...


,model,subset,horizon_hour,n_samples,MAE,RMSE,MAPE_percent
0,HA-General,test_all,1,187105,0.629436,1.032384,666.853470
1,HA-BinaryConditioned,test_all,1,187105,0.629561,1.032358,666.835187
2,HA-TypeConditioned,test_all,1,187105,0.629591,1.032372,666.830380
3,HA-General,test_all,3,187105,0.670787,1.092210,684.447307
4,HA-BinaryConditioned,test_all,3,187105,0.670827,1.092205,684.442084
5,HA-TypeConditioned,test_all,3,187105,0.670870,1.092231,684.449052
6,HA-General,test_all,6,187105,0.722277,1.159487,694.139758
7,HA-BinaryConditioned,test_all,6,187105,0.722292,1.159502,694.157471
8,HA-TypeConditioned,test_all,6,187105,0.722316,1.159523,694.141449
9,HA-General,test_incident_only,1,617,0.574602,0.934727,199.780815


## 10. Inspect learned corrections

In [23]:

for model_name, model in fitted_models.items():
    print(f"\n{model_name}")
    print(
        "Global correction [t+1, t+3, t+6]:",
        np.round(model.global_correction, 6)
    )

    if model.mode == "general":
        continue

    rows = []

    for label, correction in model.class_corrections.items():
        if model.mode == "binary":
            label_name = (
                "INCIDENT" if label == 1
                else "NO_INCIDENT"
            )
        else:
            label_name = INCIDENT_TYPE_MAP[label]

        rows.append({
            "label": label,
            "label_name": label_name,
            "train_count": model.class_counts[label],
            "correction_t+1": correction[0],
            "correction_t+3": correction[1],
            "correction_t+6": correction[2],
        })

    display(pd.DataFrame(rows))



HA-General
Global correction [t+1, t+3, t+6]: [0.001671 0.001877 0.002014]

HA-BinaryConditioned
Global correction [t+1, t+3, t+6]: [0.001671 0.001877 0.002014]


,label,label_name,train_count,correction_t+1,correction_t+3,correction_t+6
0,0,NO_INCIDENT,602069,0.001920,0.001963,0.001799
1,1,INCIDENT,1796,-0.081912,-0.026700,0.074023



HA-TypeConditioned
Global correction [t+1, t+3, t+6]: [0.001671 0.001877 0.002014]


,label,label_name,train_count,correction_t+1,correction_t+3,correction_t+6
0,0,NO_INCIDENT,602069,0.001920,0.001963,0.001799
1,1,CRASH,396,-0.014396,0.045998,0.165466
2,2,BREAKDOWN,629,-0.171774,-0.081650,0.058030
3,3,HAZARD,85,-0.239965,-0.276597,-0.187945
4,4,ROADWORK,146,-0.030640,0.006323,0.102022
5,5,TRAFFIC_CONTROL,167,-0.073892,0.003992,0.082963
6,6,ADVERSE_WEATHER,59,0.022707,0.064038,0.145892
7,7,EVENT,314,0.007974,0.010613,0.030374


## 11. Marginal value of incident information

In [24]:

def build_delta_table(metrics, subset):
    selected = metrics[metrics["subset"] == subset]

    pivot = selected.pivot_table(
        index="horizon_hour",
        columns="model",
        values="MAE",
        aggfunc="first"
    )

    required = [
        "HA-General",
        "HA-BinaryConditioned",
        "HA-TypeConditioned",
    ]

    missing = [m for m in required if m not in pivot.columns]
    if missing:
        raise ValueError(
            f"Missing models for delta calculation: {missing}"
        )

    pivot["Delta1_General_minus_Binary"] = (
        pivot["HA-General"] -
        pivot["HA-BinaryConditioned"]
    )

    pivot["Delta2_Binary_minus_Type"] = (
        pivot["HA-BinaryConditioned"] -
        pivot["HA-TypeConditioned"]
    )

    return pivot.reset_index()


delta_all = build_delta_table(metrics_df, "test_all")
delta_incident = build_delta_table(
    metrics_df,
    "test_incident_only"
)

print("All test samples")
display(delta_all)

print("Incident-only test samples")
display(delta_incident)


All test samples


model,horizon_hour,HA-BinaryConditioned,HA-General,HA-TypeConditioned,Delta1_General_minus_Binary,Delta2_Binary_minus_Type
0,1,0.629561,0.629436,0.629591,-0.000125,-0.000030
1,3,0.670827,0.670787,0.670870,-0.000040,-0.000043
2,6,0.722292,0.722277,0.722316,-0.000014,-0.000025


Incident-only test samples


model,horizon_hour,HA-BinaryConditioned,HA-General,HA-TypeConditioned,Delta1_General_minus_Binary,Delta2_Binary_minus_Type
0,1,0.587508,0.574602,0.596707,-0.012906,-0.009198
1,3,0.657344,0.653744,0.670393,-0.003601,-0.013048
2,6,0.757091,0.730970,0.764623,-0.026122,-0.007532


## 12. Final result table

In [25]:

final_table = (
    metrics_df[
        metrics_df["subset"].isin(
            ["test_all", "test_incident_only"]
        )
    ]
    .pivot_table(
        index=["model", "subset"],
        columns="horizon_hour",
        values=["MAE", "RMSE", "MAPE_percent"],
        aggfunc="first"
    )
    .round(4)
)

display(final_table)


MAE                 MAPE_percent                        RMSE                
horizon_hour                                  1       3       6            1         3         6       1       3       6
model                subset                                                                                             
HA-BinaryConditioned test_all            0.6296  0.6708  0.7223     666.8352  684.4421  694.1575  1.0324  1.0922  1.1595
                     test_incident_only  0.5875  0.6573  0.7571     197.2793  146.3637  306.9478  0.9254  1.0829  1.1814
HA-General           test_all            0.6294  0.6708  0.7223     666.8535  684.4473  694.1398  1.0324  1.0922  1.1595
                     test_incident_only  0.5746  0.6537  0.7310     199.7808  147.1935  303.4831  0.9347  1.0845  1.1767
HA-TypeConditioned   test_all            0.6296  0.6709  0.7223     666.8304  684.4491  694.1414  1.0324  1.0922  1.1595
                     test_incident_only  0.5967  0.6704  0.7646     195.8217  148.4768  302.0889  0.9304  1.0907  1.1877

## 13. Save outputs

In [ ]:

metrics_path = OUTPUT_DIR / "ha_metrics.csv"
predictions_path = OUTPUT_DIR / "ha_test_predictions.csv"
delta_all_path = OUTPUT_DIR / "ha_delta_test_all.csv"
delta_incident_path = (
    OUTPUT_DIR / "ha_delta_test_incident_only.csv"
)
audit_path = OUTPUT_DIR / "ha_dataset_audit.csv"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
delta_all.to_csv(delta_all_path, index=False)
delta_incident.to_csv(delta_incident_path, index=False)
audit_df.to_csv(audit_path, index=False)

config = {
    "train_path": TRAIN_PATH,
    "validation_path": VAL_PATH,
    "test_path": TEST_PATH,
    "n_timesteps": N_TIMESTEPS,
    "n_features": N_FEATURES,
    "horizons": HORIZONS,
    "total_flow_index": TOTAL_FLOW_IDX,
    "impact_sequence_hour_index": IMPACT_SEQUENCE_HOUR_IDX,
    "incident_type_index": INCIDENT_TYPE_IDX,
    "min_class_count": MIN_CLASS_COUNT,
    "incident_type_mapping": INCIDENT_TYPE_MAP,
}

config_path = OUTPUT_DIR / "ha_experiment_config.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Saved:")
for path in [
    metrics_path,
    predictions_path,
    delta_all_path,
    delta_incident_path,
    audit_path,
    config_path,
]:
    print("-", path)



## Important interpretation notes

- HA-General uses only the 24-hour historical total-flow average.
- HA-BinaryConditioned uses incident versus non-incident residual corrections learned from train.
- HA-TypeConditioned uses incident-category residual corrections learned from train.
- HA-TypeConditioned is not a learnable 16-dimensional embedding. It is the appropriate non-neural HA analogue for the TypeEmb experiment.
- The main TraffiDent-style comparison should include both all-test and incident-only results.
- Because the data are Z-score normalized per station, the output metrics are also in normalized units.
